# 🎥 Shorts Creator IA: Automated Podcast Highlights

Este notebook automatiza a criação de cortes virais para podcasts usando Inteligência Artificial. 

**Compatibilidade:** Google Colab, Vast.ai, RunPod e Local. 
**Foco:** Uso do `conda` para gerenciamento de ambiente, conforme o README do projeto.

## 1. Setup do Ambiente (Conda-First)
Esta célula prepara o ambiente. Se o `conda` não for detectado (como no Colab padrão), ele será instalado automaticamente sem reiniciar o kernel.

In [ ]:
# @title 🛠️ Instalar e Configurar Conda
import os
import sys
import subprocess
import shutil

def run_cmd(cmd):
    print(f"Executando: {cmd}")
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in process.stdout:
        print(line.decode().strip())

# 1. Clonar o repositório se necessário
repo_url = "https://github.com/armandocastrodesousajunior/shorts-creator-ia.git"
repo_name = "shorts-creator-ia"
if not os.path.exists(repo_name):
    run_cmd(f"git clone {repo_url}")

if os.path.exists(repo_name):
    os.chdir(repo_name)

# 2. Detectar ou Instalar Miniconda
conda_installed = shutil.which("conda") is not None

if not conda_installed:
    print("Conda não encontrado. Instalando Miniconda para Linux (Colab/Cloud)...")
    run_cmd("wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh")
    run_cmd("bash miniconda.sh -b -f -p /usr/local")
    # Garantir que o conda esteja no PATH para este processo
    sys.path.append("/usr/local/lib/python3.10/site-packages")
    conda_installed = True

# 3. Instalar dependências usando o environment.yml
if conda_installed:
    print("Atualizando ambiente com environment.yml...")
    # Usamos o ambiente 'base' no Colab para evitar conflitos de kernel
    run_cmd("conda env update -n base -f environment.yml")
    run_cmd("apt-get update && apt-get install -y ffmpeg")

print("✅ Ambiente pronto com Conda!")

## 2. Upload do Vídeo
Carregue o arquivo do podcast (.mp4) para começar.

In [ ]:
# @title 📥 Upload do Podcast (.mp4)
import sys
if 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    video_path = list(uploaded.keys())[0]
    print(f"Vídeo pronto: {video_path}")
else:
    # Tente encontrar um arquivo .mp4 na pasta
    import glob
    mp4_files = glob.glob("*.mp4")
    if mp4_files:
        video_path = mp4_files[0]
        print(f"Arquivo .mp4 detectado: {video_path}")
    else:
        video_path = "PODCAST.mp4"
        print(f"Nenhum .mp4 encontrado. Esperando: {video_path}")

## 3. Configuração da IA

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from transcription import AudioTranscriber
from visual_analysis import VisualAnalyzer
from moment_selection import MomentSelector
from video_editor import VideoEditor

# @title ⚙️ Parâmetros Básicos
MODO = "Fast" # @param ["Fast", "High Quality"]
WHISPER_MODEL = "large-v3" # @param ["tiny", "base", "small", "medium", "large-v2", "large-v3"]
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2" # @param ["mistralai/Mistral-7B-Instruct-v0.2", "meta-llama/Meta-Llama-3-8B-Instruct", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"]

print("Inicializando modelos...")
transcriber = AudioTranscriber(model_size=WHISPER_MODEL)
selector = MomentSelector(model_id=LLM_MODEL)
editor = VideoEditor()

## 4. Executar Exportação de Cortes

In [ ]:
# @title 🎬 Gerar Shorts e JSON Metadata
output_dir = "saida_cortes"
os.makedirs(output_dir, exist_ok=True)

print(f"Processando: {video_path}")
transcription = transcriber.transcribe(video_path)

print("Analisando momentos virais...")
moments = selector.select_moments(transcription)

print(f"Gerando {len(moments)} clipes...")
clip_paths = []
import json

for i, m in enumerate(moments):
    clip_name = f"short_{i}"
    mp4_out = os.path.join(output_dir, f"{clip_name}.mp4")
    json_out = os.path.join(output_dir, f"{clip_name}.json")
    
    if editor.cut_video(video_path, mp4_out, m['start'], m['end']):
        clip_paths.append(mp4_out)
        meta = {"id": i, "time": f"{m['start']}-{m['end']}", "climax": m['reason']}
        with open(json_out, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f"✅ {clip_name}.mp4 finalizado.")

if clip_paths:
    editor.merge_videos(clip_paths, os.path.join(output_dir, "compilacao.mp4"))
print("💥 TUDO PRONTO!")